# 개별종목 조합I — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5019,0.5012,0.0007,0.2995,0.3571,0.0743,0.3797,0.0527,0.1234
1,2,balanced,980,20150123,20150421,0.3953,0.3978,-0.0025,0.3495,0.3643,0.0593,0.3787,0.1857,0.2784
2,3,balanced,1210,20151228,20160328,0.3611,0.3762,-0.0151,0.3590,0.3590,0.0414,0.3710,0.3179,0.3448
3,4,balanced,1439,20161202,20170228,0.4649,0.4617,0.0032,0.3676,0.3862,0.1046,0.4080,0.1715,0.2803
4,5,balanced,1669,20171113,20180207,0.4165,0.3901,0.0265,0.3811,0.3931,0.1002,0.3936,0.2598,0.3381
5,6,balanced,1899,20181024,20190118,0.4135,0.3725,0.0411,0.4126,0.4222,0.1360,0.4185,0.5296,0.4458
6,7,balanced,2129,20190930,20191224,0.4744,0.4781,-0.0038,0.3587,0.3808,0.1002,0.4067,0.1734,0.2814
7,8,balanced,2359,20200902,20201130,0.4048,0.3476,0.0571,0.4025,0.4072,0.1111,0.4048,0.4517,0.4185
8,9,balanced,2589,20210806,20211105,0.3776,0.3916,-0.0141,0.3663,0.3830,0.0696,0.3796,0.2419,0.3153
9,10,balanced,2818,20220714,20221012,0.3478,0.3454,0.0024,0.3476,0.3514,0.0286,0.3621,0.2750,0.3195


,OOS 폴드 평균
accuracy,0.4122
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0153
macro_f1,0.3687
balanced_accuracy,0.3824
mcc,0.0840
pr_auc_macro_ovr,0.3915
down_recall,0.2734
core_harmonic_mean,0.3222


재실행 명령: python scripts/run_stock_model_experiment.py
